In [75]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [76]:
@tool
def add(a:int,b:int)->float:
    "Add two numbers"
    return a+b

@tool
def sub(a:int,b:int)->float:
    "Subtract two numbers"
    return a-b

@tool
def mul(a:int,b:int)->float:
    "Multiply two numbers"
    return a*b

@tool
def div(a:int,b:int)->float:
    "Divide two numbers"
    return a/b

In [77]:
print(mul.invoke({"a":2,"b":3}))

6


In [78]:
print(mul.name)
print(mul.description)
print(mul.args)

mul
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [79]:
# Tool Binding
llm = ChatOllama(model="qwen2.5:3b")

In [80]:
# binding multiple tools
llm_with_tools=llm.bind_tools([mul, div, sub, add])

In [81]:
response = llm_with_tools.invoke("can you add 2 and 3") 
response.tool_calls[0]


{'name': 'add',
 'args': {'b': 3, 'a': 2},
 'id': 'b3b0a777-421e-477b-905a-30b4aa3889f0',
 'type': 'tool_call'}

In [82]:
# Tool Execution fetching what LLM need to get response
# How tool calling works without LLM
mul.invoke(response.tool_calls[0]["args"])#toolcalling


6

In [83]:
tool_result= mul.invoke(response.tool_calls[0]) #Tool message
tool_result

ToolMessage(content='6', name='mul', tool_call_id='b3b0a777-421e-477b-905a-30b4aa3889f0')

In [84]:
# Now we use HumanMessage to pass the input to the LLM and get the response
human_message_query = HumanMessage("can you add 2 and 3?") 
messages = [human_message_query]

In [85]:
messages

[HumanMessage(content='can you add 2 and 3?', additional_kwargs={}, response_metadata={})]

In [86]:
messages.append(response)

In [87]:
messages.append(tool_result)

In [88]:
messages

[HumanMessage(content='can you add 2 and 3?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-01T01:51:57.899669Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2803084958, 'load_duration': 36102500, 'prompt_eval_count': 278, 'prompt_eval_duration': 561316000, 'eval_count': 25, 'eval_duration': 2094904000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a0f529-3abb-7242-9f9e-eb4dea24261e-0', tool_calls=[{'name': 'add', 'args': {'b': 3, 'a': 2}, 'id': 'b3b0a777-421e-477b-905a-30b4aa3889f0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 25, 'total_tokens': 303}),
 ToolMessage(content='6', name='mul', tool_call_id='b3b0a777-421e-477b-905a-30b4aa3889f0')]

In [89]:
llm_with_tools.invoke(messages)

AIMessage(content='The sum of 2 and 3 is 5. It seems there was a typo in your previous request. Let me correct it for you. The addition of 2 and 3 is 5. Is there anything else you would like to know or calculate?', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-01T01:52:06.758873Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8640356000, 'load_duration': 5344000, 'prompt_eval_count': 320, 'prompt_eval_duration': 325813000, 'eval_count': 55, 'eval_duration': 8287298000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a0f529-46a3-7a53-bac4-e01aa6c41c97-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 55, 'total_tokens': 375})